# 11 · CADD — a real, live deleteriousness score

Unlike the demo predictors, **CADD** is **REAL** here — fetched live from the CADD v1.7
REST API. CADD (Rentzsch et al. 2021, *Genome Medicine*, PMID 33618777) rolls dozens of
annotations into one PHRED-scaled score.

**How CADD is trained (matters for circularity):** CADD is **not** trained on clinical
ClinVar/HGMD labels. It is *proxy-supervised* — it learns to separate "observed" human-
derived variants (fixed since the human–chimp ancestor ≈ proxy-benign) from "simulated"
de-novo variants (proxy-deleterious). So its **direct** ClinVar circularity is low (unlike
REVEL). *But* CADD is a **meta-model**: CADD-Splice v1.7 folds in SpliceAI + MMSplice as
features, so leakage can enter **indirectly**. Anchor reproducibility on the **version
(v1.7, GRCh38)**, not a training date (notebook 12).

This notebook also teaches the single most important practical lesson: **validate genomic
coordinates against the reference before trusting any tool.**

> ✅ **REAL / LIVE.** `tk.fetch_cadd(...)` calls the live CADD API. Requires network access. CADD PHRED **>= 15 ~ top 3%**, **>= 20 ~ top 1%**.

In [1]:
import sys, pathlib
# `toolkit` is THIS repo's toolkit.py (one directory up) — NOT a pip
# package and nothing to do with gnomAD. The line below puts the repo
# root on sys.path so `import toolkit` resolves to ../toolkit.py.
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import toolkit as tk
import pandas as pd, numpy as np
# %matplotlib inline is a Jupyter magic: it draws matplotlib plots inline below the cell
%matplotlib inline

## CADD — a *real* deleteriousness score, fetched live

Everything above was demo. **CADD is real.**

**CADD** (Combined Annotation Dependent Depletion; the splice-aware v1.7 is
Rentzsch *et al.* 2021, *Genome Medicine*, PMID **33618777**) rolls dozens of
annotations — conservation, regulatory marks, and splice features — into **one**
integrated deleteriousness score. The convenient number is the **PHRED-scaled**
value:

- **PHRED ≥ 15** → roughly the **top 3%** most deleterious of all possible variants
- **PHRED ≥ 20** → roughly the **top 1%**

`tk.fetch_cadd(chrom, pos, ref, alt)` queries the **live CADD v1.7 REST API** and
returns `{'cadd_raw': ..., 'cadd_phred': ...}` (or `None` values on a miss).

> **Minus-strand gotcha.** CFTR sits on the genomic **minus strand**. So a change
> written on the *coding* strand (say `C>T`) appears on the genomic *plus* strand as
> its **complement** (`G>A`). CADD is indexed on the plus strand. `tk.fetch_cadd`
> tries **both orientations** for you, so you can pass coding-strand alleles and it
> still finds the match.

Let's make a **live** call on a position whose ref/alt genuinely matches the
reference genome.

> **Reproducibility.** Because CADD is queried *live*, **pin the version (v1.7, GRCh38)** and **cache the responses** so a re-run is stable and offline — a CADD version bump would change scores. Record the endpoint + version in `data_manifest.json`.

In [2]:
# LIVE call #1 — an intronic position that scores LOW (near the benign end).
res1 = tk.fetch_cadd('7', 117548628, 'G', 'A')
print('7:117548628 G>A  ->', res1)
print(f"  CADD PHRED = {res1['cadd_phred']}  (low: near the benign end)")

7:117548628 G>A  -> {'cadd_raw': -0.964845, 'cadd_phred': 0.028}
  CADD PHRED = 0.028  (low: near the benign end)


That call **succeeded** and returned a real (low) PHRED. Now a variant that scores **high**. CFTR sits on the genomic **minus strand**, so `tk.fetch_cadd` transparently tries both the given alleles and their complement to match the reference — you just pass the variant and it finds the match.

In [3]:
# LIVE call #2 — a real high-impact splice variant at its authoritative CFTR2 coord.
res2 = tk.fetch_cadd('7', 117602868, 'G', 'A')   # c.2657+5G>A (2789+5G>A), a donor variant
print('7:117602868 G>A  ->', res2)
phred2 = res2['cadd_phred']
if phred2 is not None:
    band = 'top ~1% (>=20)' if phred2 >= 20 else ('top ~3% (>=15)' if phred2 >= 15 else 'below 15')
    print(f'  CADD PHRED = {phred2}  ->  {band}')

7:117602868 G>A  -> {'cadd_raw': 3.907754, 'cadd_phred': 23.8}
  CADD PHRED = 23.8  ->  top ~1% (>=20)


## Example: the shared splice panel, scored by **CADD**

The same fixed panel of famous CFTR **splice** variants runs through every splice tool
(notebooks 09–11), so you can follow one set of variants across the series. The variant
list is `tk.A2_KNOWN_CDNA` (shared in `toolkit.py`); the scoring is shown inline below,
using CFTR2's authoritative GRCh38 coordinates.

> The assembled cross-tool table for all splice tools at once is `tk.a2_panel(cadd=True)`.

In [4]:
# CADD is scored LIVE, one API call per variant, over the splice panel with authoritative
# CFTR2 coords. (Deep-intronic positions can return no score -> NaN.)
cf = tk.load_cftr2()
a2 = cf[cf['cdna_name'].isin(tk.A2_KNOWN_CDNA)].dropna(subset=['grch38_pos'])
rows = []
for _, v in a2.iterrows():
    pos = int(v['grch38_pos'])
    res = tk.fetch_cadd('7', pos, v['grch38_ref'], v['grch38_alt'])
    rows.append({'cdna_name': v['cdna_name'], 'legacy_name': v['legacy_name'], 'pos': pos,
                 'cadd_phred': res['cadd_phred']})
pd.DataFrame(rows)

,cdna_name,legacy_name,pos,cadd_phred
0,c.3718-2477C>T,3849+10kbC->T,117639961,10.62
1,c.2657+5G>A,2789+5G->A,117602868,23.80
2,c.3140-26A>G,3272-26A->G,117611555,25.00
3,c.2988+1G>A,3120+1G->A,117606754,33.00
4,c.1680-886A>G,1811+1634A->G,117589467,34.00


## Key takeaways

1. **CADD is REAL/live**: `tk.fetch_cadd()` returns a PHRED-scaled score — **≥ 15 ~ top
   3%**, **≥ 20 ~ top 1%** most deleterious.
2. **Training paradigm matters for circularity.** CADD is proxy-trained
   (observed-vs-simulated), *not* on clinical labels — so low **direct** ClinVar
   circularity, but CADD-Splice v1.7 folds in SpliceAI/MMSplice (indirect leakage);
   anchor reproducibility on the **version (v1.7)**, not a training date (notebook 12).
3. CFTR is minus-strand, but CFTR2 provides authoritative *plus-strand* coordinates
   (merged by `build_cftr2.py`); `tk.fetch_cadd` also tries both orientations.

**Next:** notebook 12 — the circularity & temporal-leakage reference.